# LeetCode 3285 — Find Indices of Stable Mountains bằng Python

Link đề: https://leetcode.com/problems/find-indices-of-stable-mountains/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu hàng nghìn mảng ngẫu nhiên đúng ràng buộc với hàm kiểm tra theo định nghĩa). Ba ví dụ chính lấy từ đề; ví dụ nào tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` ngọn núi xếp thành hàng, `height[i]` là độ cao ngọn núi `i`; ngọn núi `i` là **ổn định** nếu ngọn núi **ngay trước nó** (nếu có) cao **hơn hẳn** `threshold`; trả về **chỉ số** của mọi ngọn núi ổn định (thứ tự tùy ý). Ngọn núi `0` **không bao giờ** ổn định.
2. Nắm các công cụ Python: **chỉ số danh sách** `height[i - 1]`, **chỉ số âm** (và cái bẫy `height[-1]`), `range(start, stop)`, `len`, `append`, **list comprehension**, `enumerate(..., start=1)`, `zip` và **cắt lát** `height[1:]`.
3. Đi từ cách **tìm núi đứng trước bằng quét cả mảng** → **duyệt một lượt từ `i = 1`** → **list comprehension** → **`zip` cặp liền kề** → **`itertools.compress`** (lọc bằng mặt nạ).
4. Hiểu mấu chốt: điều kiện của `i` chỉ phụ thuộc **một phần tử duy nhất** `height[i - 1]`, nên **không cần** nhớ gì thêm — chỉ cần một lượt duyệt `i` từ `1` đến `n - 1`.
5. Tránh các lỗi kinh điển: duyệt từ `0` rồi `height[-1]` quay vòng về **phần tử cuối**, so sánh nhầm `height[i]` thay cho `height[i - 1]`, dùng `>=` thay cho `>`, và trả **độ cao** thay vì **chỉ số**.

## 2. Đề bài & ví dụ

Có `n` ngọn núi xếp thành hàng, mỗi ngọn có một độ cao. Cho mảng nguyên `height` trong đó `height[i]` là độ cao của ngọn núi `i`, và một số nguyên `threshold`.

Ngọn núi được gọi là **ổn định** nếu ngọn núi **ngay trước nó** (**nếu tồn tại**) có độ cao **lớn hơn hẳn** `threshold`. **Lưu ý:** ngọn núi `0` **không** ổn định (vì trước nó không có ngọn nào).

Trả về mảng chứa **chỉ số của tất cả** ngọn núi ổn định, theo **thứ tự bất kỳ**.

**Ví dụ 1**

- Input: `height = [1,2,3,4,5]`, `threshold = 2`
- Output: `[3,4]`
- Giải thích: núi `3` ổn định vì `height[2] = 3 > 2`; núi `4` ổn định vì `height[3] = 4 > 2`.

**Ví dụ 2**

- Input: `height = [10,1,10,1,10]`, `threshold = 3`
- Output: `[1,3]`

**Ví dụ 3**

- Input: `height = [10,1,10,1,10]`, `threshold = 10`
- Output: `[]`

**Ràng buộc:** `2 <= n == height.length <= 100`; `1 <= height[i] <= 100`; `1 <= threshold <= 100`.

In [1]:
import random
from itertools import compress
from typing import List

# (height, threshold, kết quả kỳ vọng) — ba ví dụ lấy từ đề
examples = [
    ([1, 2, 3, 4, 5], 2, [3, 4]),
    ([10, 1, 10, 1, 10], 3, [1, 3]),
    ([10, 1, 10, 1, 10], 10, []),
]
for h, t, expected in examples:
    print(f"height={h}, threshold={t} -> kỳ vọng {expected}")

height=[1, 2, 3, 4, 5], threshold=2 -> kỳ vọng [3, 4]
height=[10, 1, 10, 1, 10], threshold=3 -> kỳ vọng [1, 3]
height=[10, 1, 10, 1, 10], threshold=10 -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Danh sách và chỉ số

Danh sách (`list`) là dãy các phần tử đánh số từ `0`. `height[0]` là phần tử đầu, `height[i - 1]` là phần tử **ngay trước** `height[i]`. `len(height)` là số phần tử, nên chỉ số hợp lệ là `0 .. len(height) - 1`. Truy cập ngoài khoảng đó gây `IndexError`.

In [2]:
height = [10, 1, 10, 1, 10]
print(height[0], height[1], height[4], len(height))     # 10 1 10 5
i = 3
print("núi", i, "cao", height[i], "- núi ngay trước cao", height[i - 1])
try:
    print(height[5])
except IndexError as e:
    print("IndexError:", e)

10 1 10 5
núi 3 cao 1 - núi ngay trước cao 10
IndexError: list index out of range


### 3.2. Chỉ số âm của Python và cái bẫy `height[-1]`

Python cho phép chỉ số **âm**: `height[-1]` là phần tử **cuối**, `height[-2]` là phần tử áp cuối. Tiện lợi, nhưng nguy hiểm: nếu bạn tính `height[i - 1]` với `i = 0` thì được `height[-1]` — **phần tử cuối cùng**, không phải "không có gì". Vì vậy đề nhấn mạnh núi `0` **không** ổn định, và ta phải **bắt đầu từ `i = 1`**.

In [3]:
height = [1, 2, 3, 4, 5]
print(height[-1], height[-2])           # 5 4
i = 0
print(height[i - 1])                    # 5 !!! (không phải "không có núi trước")

5 4
5


### 3.3. `range(start, stop)`

`range(1, 5)` sinh `1, 2, 3, 4` — gồm `start`, **không gồm** `stop`. Để duyệt `i` từ `1` đến `n - 1` ta dùng `range(1, n)` hoặc `range(1, len(height))`.

In [4]:
print(list(range(1, 5)))                       # [1, 2, 3, 4]
print(list(range(1, len([7, 8, 9]))))          # [1, 2]
print(list(range(1, 1)))                       # []  -- rỗng, không lỗi

[1, 2, 3, 4]
[1, 2]
[]


### 3.4. `append` và danh sách kết quả

Ta tạo `ans = []` rồi mỗi khi tìm được núi ổn định thì `ans.append(i)`. Nếu không có núi nào, `ans` giữ nguyên `[]` — đúng yêu cầu của ví dụ 3.

In [5]:
ans = []
for i in range(1, 5):
    if i % 2 == 0:
        ans.append(i)
print(ans)                # [2, 4]

[2, 4]


### 3.5. So sánh **lớn hơn hẳn** (`>`) và **lớn hơn hoặc bằng** (`>=`)

Đề nói "**strictly greater**" — lớn hơn hẳn — nên dùng `>`. Khi `height[i - 1] == threshold` (bằng nhau) thì **không** ổn định. Ví dụ 3 cho thấy điều này: `threshold = 10` và các núi cao `10` không tạo ra núi ổn định nào.

In [6]:
threshold = 10
for prev in [9, 10, 11]:
    print(f"prev = {prev}: prev > {threshold} -> {prev > threshold}, prev >= {threshold} -> {prev >= threshold}")

prev = 9: prev > 10 -> False, prev >= 10 -> False
prev = 10: prev > 10 -> False, prev >= 10 -> True
prev = 11: prev > 10 -> True, prev >= 10 -> True


### 3.6. List comprehension

`[biểu_thức for x in dãy if điều_kiện]` tạo danh sách mới gọn trong một dòng, tương đương vòng `for` + `if` + `append`.

In [7]:
squares_of_even = [x * x for x in range(1, 8) if x % 2 == 0]
print(squares_of_even)                  # [4, 16, 36]

# tương đương
result = []
for x in range(1, 8):
    if x % 2 == 0:
        result.append(x * x)
print(result == squares_of_even)        # True

[4, 16, 36]
True


### 3.7. `enumerate(..., start=1)`, cắt lát `height[1:]` và `zip`

- `height[1:]` là danh sách **bỏ phần tử đầu**; `height[:-1]` là danh sách **bỏ phần tử cuối**.
- `zip(a, b)` ghép từng cặp phần tử cùng vị trí và dừng khi một dãy hết.
- `zip(height, height[1:])` cho từng cặp **(phần tử trước, phần tử sau)** liền kề.
- `enumerate(dãy, start=1)` đánh số từ `1` thay vì `0` — rất hợp để lấy chỉ số của phần tử "sau".

In [8]:
height = [10, 1, 10, 1, 10]
print(height[1:], height[:-1])                           # [1, 10, 1, 10] [10, 1, 10, 1]
print(list(zip(height, height[1:])))                     # [(10, 1), (1, 10), (10, 1), (1, 10)]
for i, (prev, cur) in enumerate(zip(height, height[1:]), start=1):
    print(f"i = {i}: núi trước cao {prev}, núi i cao {cur}")

[1, 10, 1, 10] [10, 1, 10, 1]
[(10, 1), (1, 10), (10, 1), (1, 10)]
i = 1: núi trước cao 10, núi i cao 1
i = 2: núi trước cao 1, núi i cao 10
i = 3: núi trước cao 10, núi i cao 1
i = 4: núi trước cao 1, núi i cao 10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với mỗi ngọn núi `i`, ta chỉ hỏi **một câu**: "ngọn núi đứng ngay trước nó có cao hơn hẳn `threshold` không?". Không cần so sánh các núi với nhau, cũng không cần nhìn các núi xa hơn.

Tính tay ví dụ 2: `height = [10, 1, 10, 1, 10]`, `threshold = 3`.

| `i` | núi trước | `height[i-1]` | `> 3`? | ổn định? |
|---|---|---|---|---|
| 0 | không có | — | — | **không** (luôn) |
| 1 | núi 0 | 10 | có | **có** |
| 2 | núi 1 | 1 | không | không |
| 3 | núi 2 | 10 | có | **có** |
| 4 | núi 3 | 1 | không | không |

Vậy kết quả là `[1, 3]` — đúng với đề.

In [9]:
height, threshold = [10, 1, 10, 1, 10], 3
for i in range(1, len(height)):
    prev = height[i - 1]
    print(f"i = {i}: height[{i - 1}] = {prev}, {prev} > {threshold} ? {prev > threshold}")

i = 1: height[0] = 10, 10 > 3 ? True
i = 2: height[1] = 1, 1 > 3 ? False
i = 3: height[2] = 10, 10 > 3 ? True
i = 4: height[3] = 1, 1 > 3 ? False


## 5. Cách 1 — Brute Force / baseline: tìm núi đứng trước bằng cách quét cả mảng

**Ý tưởng:** làm đúng "như đọc đề". Với mỗi núi `i`, đi tìm "núi ngay trước nó" bằng cách **quét mọi `j`** và lấy `j` thỏa `j == i - 1` (nếu không tìm được, như `i = 0`, thì `previous` vẫn là `None` và núi `i` không ổn định). Cách này dùng `None` để biểu thị "không có núi trước", nên **không dính** cái bẫy chỉ số âm.

Cách này đúng nhưng **quét thừa**: để biết núi trước của `i`, ta đã biết ngay là `i - 1`, không cần tìm.

In [10]:
def stable_bruteforce(height: List[int], threshold: int) -> List[int]:
    n = len(height)
    ans = []
    for i in range(n):
        previous = None                       # chưa tìm thấy núi đứng trước
        for j in range(n):                    # quét mọi núi để tìm núi ngay trước i
            if j == i - 1:
                previous = height[j]
        if previous is not None and previous > threshold:
            ans.append(i)
    return ans


for h, t, expected in examples:
    print(stable_bruteforce(h, t), "kỳ vọng", expected)

[3, 4] kỳ vọng [3, 4]
[1, 3] kỳ vọng [1, 3]
[] kỳ vọng []


**Độ phức tạp (Cách 1).** Hai vòng lồng nhau, mỗi vòng `n` lần ⇒ **Time `O(n²)`**; chỉ dùng thêm vài biến và danh sách kết quả ⇒ **Space `O(1)`** (không tính kết quả trả về). Với `n <= 100` thì chỉ `10⁴` bước, vẫn chạy ngay. **Nút thắt:** vòng `j` tìm thứ ta đã biết sẵn (`i - 1`). Bỏ vòng đó, ta có `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Núi trước của `i` luôn là `i - 1`.** Ta truy cập thẳng `height[i - 1]` trong `O(1)`, không cần tìm. Và ta chỉ xét `i` từ `1` (núi `0` không có núi trước nên bị loại ngay).

Thuật toán một lượt:

1. `ans = []`.
2. Với `i` từ `1` đến `n - 1`: nếu `height[i - 1] > threshold` thì thêm `i` vào `ans`.
3. Trả về `ans`.

Mỗi `i` chỉ so sánh **một lần** ⇒ `n - 1` phép so sánh ⇒ `O(n)`.

In [11]:
height, threshold = [1, 2, 3, 4, 5], 2
ans = []
for i in range(1, len(height)):
    if height[i - 1] > threshold:
        ans.append(i)
print(ans)                          # [3, 4]

[3, 4]


## 7. Các cách tiếp theo

### Cách 2 — Duyệt một lượt từ `i = 1` (vòng `for` + `append`)

Giải thích từng đoạn:

- `range(1, len(height))` chạy `i = 1, 2, ..., n - 1` (bỏ núi `0`).
- `height[i - 1] > threshold` kiểm tra núi ngay trước.
- `ans.append(i)` ghi lại **chỉ số** (không phải độ cao).

In [12]:
def stable_loop(height: List[int], threshold: int) -> List[int]:
    ans = []
    for i in range(1, len(height)):
        if height[i - 1] > threshold:
            ans.append(i)
    return ans


class Solution2:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        ans = []
        for i in range(1, len(height)):
            if height[i - 1] > threshold:
                ans.append(i)
        return ans


for h, t, expected in examples:
    print(stable_loop(h, t), Solution2().stableMountains(h, t), "kỳ vọng", expected)

[3, 4] [3, 4] kỳ vọng [3, 4]
[1, 3] [1, 3] kỳ vọng [1, 3]
[] [] kỳ vọng []


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** (ngoài danh sách kết quả).

### Cách 3 — List comprehension

Cùng thuật toán, viết gọn một dòng: `[i for i in range(1, len(height)) if height[i - 1] > threshold]`. Ý nghĩa y hệt Cách 2; chỉ khác cú pháp.

In [13]:
def stable_comprehension(height: List[int], threshold: int) -> List[int]:
    return [i for i in range(1, len(height)) if height[i - 1] > threshold]


class Solution3:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        return [i for i in range(1, len(height)) if height[i - 1] > threshold]


for h, t, expected in examples:
    print(stable_comprehension(h, t), Solution3().stableMountains(h, t), "kỳ vọng", expected)

[3, 4] [3, 4] kỳ vọng [3, 4]
[1, 3] [1, 3] kỳ vọng [1, 3]
[] [] kỳ vọng []


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (ngoài kết quả).

### Cách 4 — `zip` cặp liền kề: `zip(height, height[1:])`

Thay vì tự tính `i - 1`, ta ghép **từng cặp (núi trước, núi sau)**. `enumerate(..., start=1)` cho chỉ số `i` của núi **sau**, `prev` là độ cao núi trước. Cách này nói đúng ý đề: "xét núi trước của mỗi núi" và không thể nhầm chỉ số `-1`.

Lưu ý: `height[1:]` **sao chép** mảng nên tốn thêm `O(n)` bộ nhớ (với `n <= 100` không đáng kể). Có thể dùng `itertools.islice` để tránh sao chép nhưng không cần cho bài này.

In [14]:
def stable_zip(height: List[int], threshold: int) -> List[int]:
    ans = []
    for i, (prev, _cur) in enumerate(zip(height, height[1:]), start=1):
        if prev > threshold:
            ans.append(i)
    return ans


class Solution4:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        return [i for i, prev in enumerate(height[:-1], start=1) if prev > threshold]


for h, t, expected in examples:
    print(stable_zip(h, t), Solution4().stableMountains(h, t), "kỳ vọng", expected)

[3, 4] [3, 4] kỳ vọng [3, 4]
[1, 3] [1, 3] kỳ vọng [1, 3]
[] [] kỳ vọng []


Trong `Solution4` ta thấy một cách còn gọn hơn: `height[:-1]` chính là **các núi có thể làm "núi trước"** (mọi núi trừ núi cuối). Đánh số chúng từ `1` bằng `enumerate(..., start=1)` thì số thứ tự chính là chỉ số của **núi đứng sau** — đúng thứ ta cần trả về.

**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (do cắt lát/`zip` tạo bản sao; không đáng kể khi `n <= 100`).

### Cách 5 — Lọc bằng mặt nạ `itertools.compress`

Tạo một dãy **mặt nạ** (`True`/`False`) cho mỗi núi sau: `h > threshold for h in height[:-1]` (núi trước có đủ cao không). `compress(range(1, n), mặt_nạ)` giữ lại các `i` mà mặt nạ là `True`. Cách này gợi ý một cách nghĩ khác: "**mặt nạ theo vị trí** rồi **lọc**".

In [15]:
def stable_compress(height: List[int], threshold: int) -> List[int]:
    n = len(height)
    mask = [h > threshold for h in height[:-1]]      # mask[k] ứng với núi sau có chỉ số k + 1
    return list(compress(range(1, n), mask))


class Solution5:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        return list(compress(range(1, len(height)), (h > threshold for h in height)))


h, t, _ = examples[1]
print([x > t for x in h[:-1]])                      # [True, False, True, False]
for h, t, expected in examples:
    print(stable_compress(h, t), Solution5().stableMountains(h, t), "kỳ vọng", expected)

[True, False, True, False]
[3, 4] [3, 4] kỳ vọng [3, 4]
[1, 3] [1, 3] kỳ vọng [1, 3]
[] [] kỳ vọng []


Ở `Solution5`, mặt nạ `(h > threshold for h in height)` dài `n` còn `range(1, n)` chỉ dài `n - 1`; `compress` **dừng khi một dãy hết**, nên phần tử cuối của mặt nạ bị bỏ qua đúng như mong muốn (núi cuối không là núi trước của ai).

**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (mặt nạ ở `stable_compress`) hoặc `O(1)` thêm (mặt nạ lười ở `Solution5`).

## 8. TRACE từng bước

Ta trace Cách 2 trên ví dụ 1 và ví dụ 2, rồi trace Cách 4 (`zip`) trên ví dụ 2.

In [16]:
def trace_loop(height: List[int], threshold: int) -> List[int]:
    print(f"height = {height}, threshold = {threshold}")
    ans = []
    for i in range(1, len(height)):
        prev = height[i - 1]
        stable = prev > threshold
        if stable:
            ans.append(i)
        print(f"  i = {i}: height[{i - 1}] = {prev}, {prev} > {threshold}? {stable} -> ans = {ans}")
    return ans


assert trace_loop([1, 2, 3, 4, 5], 2) == [3, 4]
assert trace_loop([10, 1, 10, 1, 10], 3) == [1, 3]

height = [1, 2, 3, 4, 5], threshold = 2
  i = 1: height[0] = 1, 1 > 2? False -> ans = []
  i = 2: height[1] = 2, 2 > 2? False -> ans = []
  i = 3: height[2] = 3, 3 > 2? True -> ans = [3]
  i = 4: height[3] = 4, 4 > 2? True -> ans = [3, 4]
height = [10, 1, 10, 1, 10], threshold = 3
  i = 1: height[0] = 10, 10 > 3? True -> ans = [1]
  i = 2: height[1] = 1, 1 > 3? False -> ans = [1]
  i = 3: height[2] = 10, 10 > 3? True -> ans = [1, 3]
  i = 4: height[3] = 1, 1 > 3? False -> ans = [1, 3]


In [17]:
def trace_zip(height: List[int], threshold: int) -> List[int]:
    print(f"height = {height}, threshold = {threshold}")
    print(f"  các cặp (trước, sau) = {list(zip(height, height[1:]))}")
    ans = []
    for i, (prev, cur) in enumerate(zip(height, height[1:]), start=1):
        stable = prev > threshold
        if stable:
            ans.append(i)
        print(f"  i = {i}: (trước = {prev}, sau = {cur}) -> trước > {threshold}? {stable} -> ans = {ans}")
    return ans


assert trace_zip([10, 1, 10, 1, 10], 3) == [1, 3]
assert trace_zip([10, 1, 10, 1, 10], 10) == []

height = [10, 1, 10, 1, 10], threshold = 3
  các cặp (trước, sau) = [(10, 1), (1, 10), (10, 1), (1, 10)]
  i = 1: (trước = 10, sau = 1) -> trước > 3? True -> ans = [1]
  i = 2: (trước = 1, sau = 10) -> trước > 3? False -> ans = [1]
  i = 3: (trước = 10, sau = 1) -> trước > 3? True -> ans = [1, 3]
  i = 4: (trước = 1, sau = 10) -> trước > 3? False -> ans = [1, 3]
height = [10, 1, 10, 1, 10], threshold = 10
  các cặp (trước, sau) = [(10, 1), (1, 10), (10, 1), (1, 10)]
  i = 1: (trước = 10, sau = 1) -> trước > 10? False -> ans = []
  i = 2: (trước = 1, sau = 10) -> trước > 10? False -> ans = []
  i = 3: (trước = 10, sau = 1) -> trước > 10? False -> ans = []
  i = 4: (trước = 1, sau = 10) -> trước > 10? False -> ans = []


Bảng trace Cách 2 với `height = [1, 2, 3, 4, 5]`, `threshold = 2`:

| `i` | `height[i-1]` | `> 2`? | `ans` sau bước |
|---|---|---|---|
| 1 | 1 | không | `[]` |
| 2 | 2 | không (bằng, không **hơn hẳn**) | `[]` |
| 3 | 3 | có | `[3]` |
| 4 | 4 | có | `[3, 4]` |

Kết quả `[3, 4]`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (vòng `for` rõ ràng, dễ đọc, dễ debug).

In [18]:
class Solution:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        ans = []
        for i in range(1, len(height)):          # bắt đầu từ 1: núi 0 không bao giờ ổn định
            if height[i - 1] > threshold:        # núi ngay trước cao hơn hẳn ngưỡng
                ans.append(i)                    # ghi lại CHỈ SỐ
        return ans


sol = Solution()
for h, t, expected in examples:
    print(sol.stableMountains(h, t), "kỳ vọng", expected)

[3, 4] kỳ vọng [3, 4]
[1, 3] kỳ vọng [1, 3]
[] kỳ vọng []


## 10. Vì sao thuật toán đúng

- **Núi `0` bị loại đúng:** theo đề, núi `0` không ổn định; ta bắt đầu từ `i = 1` nên không bao giờ thêm `0` vào `ans`, và cũng không bao giờ chạm `height[-1]`.
- **Điều kiện khớp định nghĩa:** với `i >= 1`, núi ngay trước chính là núi `i - 1`, nên "núi trước cao hơn hẳn `threshold`" là `height[i - 1] > threshold` — đúng nguyên văn đề, dùng `>` chứ không phải `>=`.
- **Đủ và không thừa:** mọi núi `i` từ `1` đến `n - 1` đều được xét đúng một lần; chỉ những `i` thỏa điều kiện mới vào `ans`.
- **Các điều kiện độc lập:** núi `i` có ổn định hay không **không phụ thuộc** vào việc núi khác ổn định, nên không có tương tác giữa các lần xét.
- **Thứ tự:** đề cho phép bất kỳ thứ tự; thuật toán trả theo thứ tự tăng dần của chỉ số, hợp lệ.
- Kiểm chứng thực nghiệm: Mục 12 so mọi cách với **hàm kiểm tra theo định nghĩa** trên nhiều mảng ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(height)` (`n <= 100`).

- **Cách 1 (quét tìm núi trước):** `O(n²)` thời gian, `O(1)` bộ nhớ phụ.
- **Cách 2, 3 (một lượt):** `O(n)` thời gian — mỗi `i` đúng một phép so sánh; `O(1)` bộ nhớ phụ (không tính mảng kết quả).
- **Cách 4, 5 (`zip`/cắt lát/mặt nạ):** `O(n)` thời gian; có thể tốn `O(n)` bộ nhớ phụ nếu tạo bản sao.
- Không thể nhanh hơn `O(n)` về lý thuyết vì cần đọc các `height[i - 1]` (ít nhất `n - 1` phần tử) để biết kết quả. Kết quả có thể dài tới `n - 1` phần tử.

## 12. Bộ test

Vì đề cho phép **thứ tự bất kỳ**, ta không so sánh bằng `==` một cách máy móc mà dùng **hàm kiểm tra tính hợp lệ** `is_valid_answer`: kết quả phải (1) không trùng, (2) mọi chỉ số nằm trong `[0, n)`, (3) đúng là **tập** các `i >= 1` thỏa `height[i - 1] > threshold` (tính độc lập bằng định nghĩa). Với các ví dụ đề ta kiểm cả `sorted(kết quả) == sorted(kỳ vọng)`.

In [19]:
def is_valid_answer(height, threshold, ans):
    """Kiểm tra độc lập theo định nghĩa: không phụ thuộc cách làm."""
    n = len(height)
    if len(set(ans)) != len(ans):                          # không trùng
        return False
    if any(not (0 <= i < n) for i in ans):                 # chỉ số hợp lệ
        return False
    for i in range(n):
        previous_exists = i - 1 >= 0
        is_stable = previous_exists and height[i - 1] > threshold
        if is_stable != (i in ans):
            return False
    return True


sol = Solution()
impls = {
    "bruteforce": stable_bruteforce,
    "loop": stable_loop,
    "comprehension": stable_comprehension,
    "zip": stable_zip,
    "compress": stable_compress,
    "sol2": Solution2().stableMountains,
    "sol3": Solution3().stableMountains,
    "sol4": Solution4().stableMountains,
    "sol5": Solution5().stableMountains,
    "leetcode": sol.stableMountains,
}


def check_all(height, threshold, expected, label):
    for key, fn in impls.items():
        actual = fn(list(height), threshold)                # truyền bản sao để chắc chắn không bị sửa
        assert is_valid_answer(height, threshold, actual), (label, key, height, threshold, actual)
        if expected is not None:
            assert sorted(actual) == sorted(expected), (label, key, actual, expected)
    print("PASS", label)


for idx, (h, t, expected) in enumerate(examples, 1):
    check_all(h, t, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

`n = 2` (nhỏ nhất), mọi núi trước đều cao hơn ngưỡng, không núi nào, đúng bằng ngưỡng (kiểm tra `>` hay `>=`), ngưỡng nhỏ nhất/lớn nhất (đều tự dựng, hợp lệ theo ràng buộc).

In [20]:
check_all([1, 1], 1, [], "n = 2, bằng ngưỡng (không hơn hẳn)")
check_all([5, 1], 4, [1], "n = 2, núi 0 cao hơn ngưỡng")
check_all([5, 1], 5, [], "n = 2, núi 0 bằng ngưỡng")
check_all([100, 100, 100, 100], 1, [1, 2, 3], "mọi núi cao, chỉ số 0 không có mặt")
check_all([1, 1, 1, 1, 1], 1, [], "mọi núi thấp bằng ngưỡng")
check_all([1, 2, 3, 4, 5], 100, [], "ngưỡng lớn nhất, không núi nào")
check_all([100, 1, 1, 1, 1], 99, [1], "chỉ núi 0 cao, chỉ núi 1 ổn định")
check_all([1, 1, 1, 1, 100], 1, [], "chỉ núi cuối cao nhưng không có núi sau nó")
check_all([5, 6, 7, 8], 5, [2, 3], "giá trị bằng ngưỡng (5) không tính, 6 và 7 tính")

PASS n = 2, bằng ngưỡng (không hơn hẳn)
PASS n = 2, núi 0 cao hơn ngưỡng
PASS n = 2, núi 0 bằng ngưỡng
PASS mọi núi cao, chỉ số 0 không có mặt
PASS mọi núi thấp bằng ngưỡng
PASS ngưỡng lớn nhất, không núi nào
PASS chỉ núi 0 cao, chỉ núi 1 ổn định
PASS chỉ núi cuối cao nhưng không có núi sau nó
PASS giá trị bằng ngưỡng (5) không tính, 6 và 7 tính


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách sai: (a) **duyệt từ `0`** (dính `height[-1]`), (b) **so sánh `height[i]`** thay vì `height[i - 1]`, (c) **dùng `>=`**. Ta tìm đầu vào làm chúng lộ sai.

In [21]:
def wrong_from_zero(height, threshold):
    return [i for i in range(len(height)) if height[i - 1] > threshold]     # i = 0 -> height[-1]

def wrong_current(height, threshold):
    return [i for i in range(1, len(height)) if height[i] > threshold]

def wrong_ge(height, threshold):
    return [i for i in range(1, len(height)) if height[i - 1] >= threshold]


cases = [([1, 2, 3, 4, 5], 2), ([10, 1, 10, 1, 10], 3), ([10, 1, 10, 1, 10], 10), ([1, 1, 1, 9], 5)]
for name, wrong in [("duyệt từ 0", wrong_from_zero), ("height[i]", wrong_current), (">= thay vì >", wrong_ge)]:
    for h, t in cases:
        if not is_valid_answer(h, t, wrong(h, t)):
            print(f"cách '{name}' sai trên height={h}, threshold={t}: trả {wrong(h, t)}, đúng là {sol.stableMountains(h, t)}")
            break
    else:
        raise AssertionError(name)

cách 'duyệt từ 0' sai trên height=[1, 2, 3, 4, 5], threshold=2: trả [0, 3, 4], đúng là [3, 4]
cách 'height[i]' sai trên height=[1, 2, 3, 4, 5], threshold=2: trả [2, 3, 4], đúng là [3, 4]
cách '>= thay vì >' sai trên height=[1, 2, 3, 4, 5], threshold=2: trả [2, 3, 4], đúng là [3, 4]


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `5000` bộ `(height, threshold)` thỏa ràng buộc đề: `2 <= n <= 100`, `1 <= height[i] <= 100`, `1 <= threshold <= 100`. Một nửa số bộ dùng giá trị nhỏ (`1..5`) để va chạm `>`/`>=` nhiều hơn. Mỗi kết quả của mọi cách phải qua `is_valid_answer`, và các cách khớp nhau.

In [22]:
rng = random.Random(3285)
for trial in range(5000):
    n = rng.choice([2, 3, 4, 5, 10, 50, 99, 100])
    hi = 5 if trial % 2 == 0 else 100
    height = [rng.randint(1, hi) for _ in range(n)]
    threshold = rng.randint(1, hi)
    assert 2 <= len(height) <= 100 and all(1 <= x <= 100 for x in height) and 1 <= threshold <= 100
    ref = stable_loop(height, threshold)
    assert is_valid_answer(height, threshold, ref)
    for key, fn in impls.items():
        got = fn(list(height), threshold)
        assert is_valid_answer(height, threshold, got), (key, height, threshold)
        assert sorted(got) == ref, (key, height, threshold)
print("PASS 5000 bộ ngẫu nhiên đúng ràng buộc")

PASS 5000 bộ ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt từ `0` rồi dùng `height[i - 1]` (cái bẫy `height[-1]`)

In [23]:
def sai_tu_0(height, threshold):
    ans = []
    for i in range(len(height)):
        if height[i - 1] > threshold:          # i = 0 -> height[-1] = phần tử CUỐI
            ans.append(i)
    return ans

print(sai_tu_0([10, 1, 1, 1, 10], 3))           # [0, 1]  -- núi 0 KHÔNG được phép ổn định
print("đúng là", Solution().stableMountains([10, 1, 1, 1, 10], 3))   # [1]

[0, 1]
đúng là [1]


**Vì sao sai:** với `i = 0`, `height[i - 1]` là `height[-1]` — **phần tử cuối** — nên núi `0` bị xét ổn định chỉ vì ngọn núi cuối mảng cao. Python **không báo lỗi**, kết quả sai âm thầm. Hãy bắt đầu `range(1, n)`.

### Lỗi 2: So sánh nhầm `height[i]` (núi hiện tại) thay vì núi trước

In [24]:
def sai_nham_chi_so(height, threshold):
    ans = []
    for i in range(1, len(height)):
        if height[i] > threshold:              # phải là height[i - 1]
            ans.append(i)
    return ans

print(sai_nham_chi_so([1, 2, 3, 4, 5], 2), "(đúng là [3, 4])")

[2, 3, 4] (đúng là [3, 4])


**Vì sao sai:** đề hỏi về **núi đứng trước**, không phải chính núi `i`. Với ví dụ 1 ta thu được `[2, 3, 4]` thay vì `[3, 4]`. Mẹo: đọc to điều kiện — "độ cao của núi **trước** `i` lớn hơn hẳn ngưỡng" ⇒ `height[i - 1]`.

### Lỗi 3: Dùng `>=` thay cho `>`

In [25]:
def sai_bang(height, threshold):
    return [i for i in range(1, len(height)) if height[i - 1] >= threshold]

print(sai_bang([10, 1, 10, 1, 10], 10), "(đúng là [])")

[1, 3] (đúng là [])


**Vì sao sai:** "strictly greater" là **lớn hơn hẳn**; bằng ngưỡng thì **không** ổn định (ví dụ 3 minh họa). Luôn kiểm tra đúng ở **ví dụ có giá trị bằng ngưỡng**.

### Lỗi 4: Trả về độ cao thay vì chỉ số

In [26]:
def sai_tra_do_cao(height, threshold):
    ans = []
    for i in range(1, len(height)):
        if height[i - 1] > threshold:
            ans.append(height[i])              # sai: phải là i
    return ans

print(sai_tra_do_cao([1, 2, 3, 4, 5], 2), "(đúng là [3, 4]: chỉ số)")      # [4, 5]

[4, 5] (đúng là [3, 4]: chỉ số)


**Vì sao sai:** đề yêu cầu **chỉ số** của các núi ổn định. Ở ví dụ 1 kết quả `[4, 5]` (độ cao) khác `[3, 4]` (chỉ số). Khi test, hãy chọn đầu vào mà **độ cao khác chỉ số** để lỗi lộ ra.

**Code đúng (nhắc lại):** `[i for i in range(1, len(height)) if height[i - 1] > threshold]`.

In [27]:
def dung(height, threshold):
    return [i for i in range(1, len(height)) if height[i - 1] > threshold]

for h, t, expected in examples:
    assert sorted(dung(h, t)) == sorted(expected)
print("PASS code đúng")

PASS code đúng


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "**phần tử ngay trước/ngay sau**" ⇒ nghĩ tới duyệt `i` kèm `i - 1`/`i + 1`, **và** nhớ xét biên: `i = 0` không có phần tử trước, `i = n - 1` không có phần tử sau.
- Cẩn thận `a[i - 1]` khi `i = 0`: Python trả về **phần tử cuối** thay vì báo lỗi. Cách an toàn: bắt đầu `range(1, n)`, hoặc `if i > 0 and ...`.
- Điều kiện mỗi phần tử **độc lập** nhau ⇒ một lượt duyệt là đủ, không cần cấu trúc dữ liệu thêm.
- "**Trả về chỉ số**" ⇒ dùng `i`, không phải `a[i]`. Đừng quên chọn test mà chỉ số khác giá trị.
- "**Strictly greater**" ⇒ `>`; "at least" ⇒ `>=`. Luôn thử test đúng bằng ngưỡng.
- "**Thứ tự bất kỳ**" ⇒ khi tự test, so sánh `sorted(...)` hoặc dùng hàm kiểm tra hợp lệ.

## 15. Biến thể của bài

1. **Đếm** số núi ổn định (chỉ cần `len` hoặc `sum`).
2. **Hai phía:** núi ổn định nếu **ít nhất một** núi kề (trái hoặc phải) cao hơn hẳn ngưỡng.
3. **Khoảng cách `k`:** núi `i` ổn định nếu núi `i - k` cao hơn hẳn ngưỡng.
4. **Trả về độ cao** của các núi ổn định thay vì chỉ số.
5. **Mặt nạ boolean** `stable[i]` cho mọi núi (kể cả núi `0` luôn `False`).

In [28]:
def count_stable(height, threshold):
    return sum(1 for i in range(1, len(height)) if height[i - 1] > threshold)


def stable_either_side(height, threshold):
    n = len(height)
    ans = []
    for i in range(n):
        left = i > 0 and height[i - 1] > threshold
        right = i < n - 1 and height[i + 1] > threshold
        if left or right:
            ans.append(i)
    return ans


def stable_k_back(height, threshold, k):
    return [i for i in range(k, len(height)) if height[i - k] > threshold]


def stable_heights(height, threshold):
    return [height[i] for i in range(1, len(height)) if height[i - 1] > threshold]


def stable_mask(height, threshold):
    return [i > 0 and height[i - 1] > threshold for i in range(len(height))]


h = [10, 1, 10, 1, 10]
print(count_stable(h, 3))                      # 2
print(stable_either_side(h, 3))                # [1, 3]
print(stable_k_back(h, 3, 2))                  # [2, 4]
print(stable_heights(h, 3))                    # [1, 1]
print(stable_mask(h, 3))                       # [False, True, False, True, False]

2
[1, 3]
[2, 4]
[1, 1]
[False, True, False, True, False]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tìm núi trước | với mỗi `i`, duyệt `j` tìm `j == i - 1` | `O(n²)` | `O(1)` | bám sát đề thô, chỉ để học |
| 2. Một lượt `for` | `for i in range(1, n)` + `append` | `O(n)` | `O(1)` | **nên nộp**, dễ debug |
| 3. List comprehension | cùng ý, viết một dòng | `O(n)` | `O(1)` | gọn, quen tay |
| 4. `zip` / `enumerate(height[:-1], 1)` | duyệt cặp (trước, sau) | `O(n)` | `O(n)` | tránh sai chỉ số `-1`, đọc như đề |
| 5. `compress` + mặt nạ | lọc `range` bằng dãy `True/False` | `O(n)` | `O(n)` hoặc `O(1)` | khi quen `itertools`, nghĩ theo mặt nạ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[7, 8, 9][-1]`, `list(range(1, 4))`, `[4, 5, 6][1:]`, `Solution().stableMountains([5, 5], 4)`, `Solution().stableMountains([5, 5], 5)`.

**Bài 2 (tính tay).** Với `height = [4, 9, 2, 7, 7, 1]`, `threshold = 6`: lập bảng `i`, `height[i-1]`, `> 6`?, và cho kết quả.

**Bài 3 (điền TODO).** Hoàn thành `stable_todo(height, threshold)` theo Cách 2.

In [29]:
def stable_todo(height, threshold):
    ans = []
    # TODO 1: duyệt i từ 1 đến len(height) - 1 (nhớ: núi 0 không bao giờ ổn định)
    # TODO 2: nếu height[i - 1] > threshold thì ans.append(i)
    # TODO 3: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `stable_zip2(height, threshold)` bằng `zip(height, height[1:])` và `enumerate(..., start=1)` trong **một list comprehension**.

**Bài 5 (biến thể / mini challenge).** (a) Viết `stable_either(height, threshold)`: núi `i` ổn định nếu **ít nhất một** núi kề (trái hoặc phải, nếu tồn tại) cao hơn hẳn `threshold`. (b) Viết `longest_stable_run(height, threshold)` trả về độ dài của **đoạn liên tiếp dài nhất** các núi ổn định (không có núi nào: trả `0`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `9`, `[1, 2, 3]`, `[5, 6]`, `[1]` (`5 > 4`), `[]` (`5` không lớn hơn hẳn `5`). Bài 3: `for i in range(1, len(height)):`.
- **Mức 2 (vừa):** Bài 2: `i = 1`: `height[0] = 4` không; `i = 2`: `9` có; `i = 3`: `2` không; `i = 4`: `7` có; `i = 5`: `7` có ⇒ `[2, 4, 5]`. Bài 4: `[i for i, (prev, _) in enumerate(zip(height, height[1:]), start=1) if prev > threshold]`.
- **Mức 3 (sâu):** Bài 5(a): với mỗi `i`, kiểm tra `i > 0 and height[i - 1] > threshold` **hoặc** `i < n - 1 and height[i + 1] > threshold`. 5(b): duyệt `i`, giữ biến `run` (độ dài hiện tại) và `best`; núi ổn định thì `run += 1` và cập nhật `best`, ngược lại `run = 0`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert [7, 8, 9][-1] == 9
assert list(range(1, 4)) == [1, 2, 3]
assert [4, 5, 6][1:] == [5, 6]
assert Solution().stableMountains([5, 5], 4) == [1]
assert Solution().stableMountains([5, 5], 5) == []

# Bài 2
assert Solution().stableMountains([4, 9, 2, 7, 7, 1], 6) == [2, 4, 5]


# Bài 3
def stable_todo_ans(height, threshold):
    ans = []
    for i in range(1, len(height)):
        if height[i - 1] > threshold:
            ans.append(i)
    return ans


for h, t, expected in examples:
    assert sorted(stable_todo_ans(h, t)) == sorted(expected)


# Bài 4
def stable_zip2(height, threshold):
    return [i for i, (prev, _) in enumerate(zip(height, height[1:]), start=1) if prev > threshold]


for h, t, expected in examples:
    assert sorted(stable_zip2(h, t)) == sorted(expected)


# Bài 5(a)
def stable_either(height, threshold):
    n = len(height)
    return [
        i for i in range(n)
        if (i > 0 and height[i - 1] > threshold) or (i < n - 1 and height[i + 1] > threshold)
    ]


assert stable_either([10, 1, 1, 10, 1], 3) == [1, 2, 4]
assert stable_either([1, 1, 1], 5) == []
assert stable_either([1, 9, 1, 1], 5) == [0, 2]


# Bài 5(b)
def longest_stable_run(height, threshold):
    best = run = 0
    for i in range(len(height)):
        if i > 0 and height[i - 1] > threshold:
            run += 1
            best = max(best, run)
        else:
            run = 0
    return best


assert longest_stable_run([1, 2, 3, 4, 5], 2) == 2
assert longest_stable_run([10, 1, 10, 1, 10], 3) == 1
assert longest_stable_run([10, 1, 10, 1, 10], 10) == 0
assert longest_stable_run([9, 9, 9, 9], 1) == 3
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt từ `1`, xét phần tử ngay trước, ghi **chỉ số**)

```python
class Solution:
    def stableMountains(self, height: List[int], threshold: int) -> List[int]:
        ans = []
        for i in range(1, len(height)):
            if height[i - 1] > threshold:
                ans.append(i)
        return ans
```

**Checklist**

- [ ] Vòng lặp bắt đầu từ `i = 1` (không dính `height[-1]`)?
- [ ] So sánh `height[i - 1]` (núi **trước**), không phải `height[i]`?
- [ ] Dùng `>` (lớn hơn **hẳn**), không phải `>=`?
- [ ] Ghi **chỉ số** `i`, không phải độ cao?
- [ ] Trả `[]` khi không có núi nào ổn định?
- [ ] Đã thử ba ví dụ đề (`[3,4]`, `[1,3]`, `[]`) và biên `n = 2`, giá trị bằng ngưỡng?
- [ ] Skeleton đúng `class Solution` / `def stableMountains(self, height: List[int], threshold: int) -> List[int]`?